# AI Support Chat Analyzer — Dataset Analysis

Step-by-step analysis of generated support dialogs and quality evaluation results.
This notebook loads the generated dataset and analysis results, then performs:
- Dataset overview and statistics
- Quality score distribution
- Satisfaction analysis
- Anomaly detection (hidden dissatisfaction, score inconsistencies)
- Agent mistake patterns
- Confidence score calibration
- Validation warnings analysis

In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Load Data

In [ ]:
import sys
from pathlib import Path

chats_path = Path('data/chats.json')
analysis_path = Path('results/analysis.json')

if not chats_path.exists():
    print(f"ERROR: {chats_path} not found. Run 'python generate.py' first.")
    sys.exit(1)
if not analysis_path.exists():
    print(f"ERROR: {analysis_path} not found. Run 'python analyze.py' first.")
    sys.exit(1)

# Load generated dialogs
with open(chats_path, 'r', encoding='utf-8') as f:
    chats_data = json.load(f)

# Load analysis results
with open(analysis_path, 'r', encoding='utf-8') as f:
    analysis_data = json.load(f)

chats = chats_data['chats']
results = analysis_data['results']

print(f"Dataset metadata: {chats_data['metadata']}")
print(f"Analysis metadata: {analysis_data['metadata']}")
print(f"\nTotal dialogs: {len(chats)}")
print(f"Total analysis results: {len(results)}")

In [ ]:
# Build DataFrames
df_results = pd.DataFrame(results)

# Extract scenario info from chats
chat_scenarios = []
for chat in chats:
    scenario = chat['scenario']
    chat_scenarios.append({
        'chat_id': chat['id'],
        'category': scenario['category'],
        'case_type': scenario['case_type'],
        'has_hidden_dissatisfaction': scenario.get('has_hidden_dissatisfaction', False),
        'intended_mistakes': scenario.get('intended_agent_mistakes', []),
        'num_messages': len(chat['messages']),
    })

df_scenarios = pd.DataFrame(chat_scenarios)

# Merge scenarios with analysis results
df = df_scenarios.merge(df_results, on='chat_id', how='inner')
print(f"Merged dataset: {len(df)} rows")
df.head()

## 2. Dataset Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Category distribution
df['category'].value_counts().plot(kind='barh', ax=axes[0], color=sns.color_palette('muted'))
axes[0].set_title('Distribution by Category')
axes[0].set_xlabel('Count')

# Case type distribution
df['case_type'].value_counts().plot(kind='barh', ax=axes[1], color=sns.color_palette('muted'))
axes[1].set_title('Distribution by Case Type')
axes[1].set_xlabel('Count')

# Dialog length distribution
axes[2].hist(df['num_messages'], bins=range(4, 22), edgecolor='black', alpha=0.7)
axes[2].set_title('Dialog Length (number of messages)')
axes[2].set_xlabel('Messages')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f"Average dialog length: {df['num_messages'].mean():.1f} messages")
print(f"Min: {df['num_messages'].min()}, Max: {df['num_messages'].max()}")

## 3. Quality Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall quality score distribution
score_counts = df['quality_score'].value_counts().sort_index()
colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']
score_counts.plot(kind='bar', ax=axes[0], color=colors)
axes[0].set_title('Quality Score Distribution')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(score_counts.index, rotation=0)

# Quality score by case type
df.boxplot(column='quality_score', by='case_type', ax=axes[1])
axes[1].set_title('Quality Score by Case Type')
axes[1].set_xlabel('Case Type')
axes[1].set_ylabel('Quality Score')
plt.suptitle('')

plt.tight_layout()
plt.show()

print(f"Average quality score: {df['quality_score'].mean():.2f}")
print(f"Median: {df['quality_score'].median()}")
print(f"Std dev: {df['quality_score'].std():.2f}")
print("\nScore distribution:")
for score in range(1, 6):
    count = (df['quality_score'] == score).sum()
    pct = count / len(df) * 100
    print(f"  Score {score}: {count:3d} ({pct:.1f}%)")

## 4. Satisfaction Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Satisfaction pie chart
sat_counts = df['satisfaction'].value_counts()
sat_colors = {'satisfied': '#2ecc71', 'neutral': '#f1c40f', 'unsatisfied': '#e74c3c'}
ordered_colors = [sat_colors[level] for level in sat_counts.index]
sat_counts.plot(kind='pie', ax=axes[0], autopct='%1.1f%%', colors=ordered_colors,
                startangle=90, textprops={'fontsize': 11})
axes[0].set_title('Client Satisfaction Distribution')
axes[0].set_ylabel('')

# Satisfaction by category
ct = pd.crosstab(df['category'], df['satisfaction'], normalize='index') * 100
ct_ordered = ct[['satisfied', 'neutral', 'unsatisfied']] if 'satisfied' in ct.columns else ct
ct_ordered.plot(kind='barh', stacked=True, ax=axes[1],
                color=['#2ecc71', '#f1c40f', '#e74c3c'])
axes[1].set_title('Satisfaction by Category (%)')
axes[1].set_xlabel('Percentage')
axes[1].legend(title='Satisfaction')

plt.tight_layout()
plt.show()

## 5. Anomaly Detection

### 5.1 Hidden Dissatisfaction Detection

Compare intended hidden dissatisfaction (from scenario) with detected dissatisfaction (from analysis).

In [ ]:
# Hidden dissatisfaction analysis
hidden = df[df['has_hidden_dissatisfaction']].copy()
non_hidden = df[~df['has_hidden_dissatisfaction']].copy()

print(f"Dialogs with intended hidden dissatisfaction: {len(hidden)}")
print(f"Dialogs without hidden dissatisfaction: {len(non_hidden)}")

# Detection accuracy: hidden dissatisfaction should be detected as 'unsatisfied' or 'neutral'
# (both count as detection — the key is that the model does NOT mark them as 'satisfied')
hidden_detected = (hidden['satisfaction'] != 'satisfied').sum()
hidden_accuracy = hidden_detected / len(hidden) * 100 if len(hidden) > 0 else 0

print("\nHidden dissatisfaction detection:")
print(f"  Correctly detected (unsatisfied/neutral): {hidden_detected}/{len(hidden)} ({hidden_accuracy:.1f}%)")
print(f"  Missed (marked as satisfied): {len(hidden) - hidden_detected}")

# Breakdown of detection results
print("\n  Detection breakdown:")
for level in ['unsatisfied', 'neutral', 'satisfied']:
    count = (hidden['satisfaction'] == level).sum()
    print(f"    {level}: {count}")

# False positives: non-hidden dialogs marked as unsatisfied
false_positives = (non_hidden['satisfaction'] == 'unsatisfied').sum()
print(f"\n  False positives (non-hidden marked unsatisfied): {false_positives}/{len(non_hidden)}")

# Show missed cases
missed = hidden[hidden['satisfaction'] == 'satisfied']
if len(missed) > 0:
    print("\nMissed hidden dissatisfaction cases:")
    for _, row in missed.iterrows():
        print(f"  {row['chat_id']}: satisfaction={row['satisfaction']}, "
              f"score={row['quality_score']}, category={row['category']}")

In [ ]:
# Visualize hidden dissatisfaction
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Satisfaction comparison: hidden vs non-hidden
labels = ['satisfied', 'neutral', 'unsatisfied']
hidden_sat = hidden['satisfaction'].value_counts().reindex(labels, fill_value=0)
non_hidden_sat = non_hidden['satisfaction'].value_counts().reindex(labels, fill_value=0)

x = range(len(labels))
width = 0.35
axes[0].bar([i - width/2 for i in x], hidden_sat.values, width, label='Hidden dissatisfaction', color='#e74c3c', alpha=0.8)
axes[0].bar([i + width/2 for i in x], non_hidden_sat.values, width, label='No hidden dissatisfaction', color='#3498db', alpha=0.8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels)
axes[0].set_title('Satisfaction: Hidden vs Non-Hidden')
axes[0].set_ylabel('Count')
axes[0].legend()

# Quality score comparison
axes[1].hist(hidden['quality_score'], bins=range(1, 7), alpha=0.6, label='Hidden dissatisfaction', color='#e74c3c', edgecolor='black')
axes[1].hist(non_hidden['quality_score'], bins=range(1, 7), alpha=0.6, label='No hidden dissatisfaction', color='#3498db', edgecolor='black')
axes[1].set_title('Quality Score: Hidden vs Non-Hidden')
axes[1].set_xlabel('Quality Score')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

### 5.2 Score Inconsistencies

Detect anomalies where quality score contradicts satisfaction level.

In [ ]:
# Anomaly 1: High score but unsatisfied client
high_score_unsatisfied = df[(df['quality_score'] >= 4) & (df['satisfaction'] == 'unsatisfied')]
print(f"Anomaly: High score (>=4) but client unsatisfied: {len(high_score_unsatisfied)} cases")
if len(high_score_unsatisfied) > 0:
    for _, row in high_score_unsatisfied.iterrows():
        print(f"  {row['chat_id']}: score={row['quality_score']}, category={row['category']}, "
              f"case_type={row['case_type']}, summary={row['summary'][:80]}...")

print()

# Anomaly 2: Low score but satisfied client
low_score_satisfied = df[(df['quality_score'] <= 2) & (df['satisfaction'] == 'satisfied')]
print(f"Anomaly: Low score (<=2) but client satisfied: {len(low_score_satisfied)} cases")
if len(low_score_satisfied) > 0:
    for _, row in low_score_satisfied.iterrows():
        print(f"  {row['chat_id']}: score={row['quality_score']}, category={row['category']}, "
              f"case_type={row['case_type']}, summary={row['summary'][:80]}...")

print()

# Anomaly 3: No mistakes but low score
no_mistakes_low = df[(df['agent_mistakes'].apply(len) == 0) & (df['quality_score'] <= 2)]
print(f"Anomaly: No agent mistakes but low score (<=2): {len(no_mistakes_low)} cases")
if len(no_mistakes_low) > 0:
    for _, row in no_mistakes_low.iterrows():
        print(f"  {row['chat_id']}: score={row['quality_score']}, category={row['category']}")

In [ ]:
# Heatmap: quality score vs satisfaction
ct_score_sat = pd.crosstab(df['quality_score'], df['satisfaction'])
ct_score_sat = ct_score_sat[['satisfied', 'neutral', 'unsatisfied']]

plt.figure(figsize=(8, 5))
sns.heatmap(ct_score_sat, annot=True, fmt='d', cmap='YlOrRd', linewidths=0.5)
plt.title('Quality Score vs Satisfaction (count)')
plt.xlabel('Satisfaction')
plt.ylabel('Quality Score')
plt.tight_layout()
plt.show()

## 6. Agent Mistake Patterns

In [ ]:
# Flatten agent mistakes
all_mistakes = []
for _, row in df.iterrows():
    for mistake in row['agent_mistakes']:
        all_mistakes.append({
            'chat_id': row['chat_id'],
            'mistake': mistake,
            'quality_score': row['quality_score'],
            'satisfaction': row['satisfaction'],
            'category': row['category'],
        })

df_mistakes = pd.DataFrame(all_mistakes)

chats_with_mistakes = (df['agent_mistakes'].apply(len) > 0).sum()
chats_without_mistakes = (df['agent_mistakes'].apply(len) == 0).sum()

print(f"Dialogs with agent mistakes: {chats_with_mistakes} ({chats_with_mistakes/len(df)*100:.1f}%)")
print(f"Dialogs without mistakes: {chats_without_mistakes} ({chats_without_mistakes/len(df)*100:.1f}%)")
print(f"Total mistake occurrences: {len(df_mistakes)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Mistake frequency
if len(df_mistakes) > 0:
    mistake_counts = df_mistakes['mistake'].value_counts()
    mistake_counts.plot(kind='barh', ax=axes[0], color=sns.color_palette('Reds_r', len(mistake_counts)))
    axes[0].set_title('Agent Mistake Frequency')
    axes[0].set_xlabel('Count')

    # Average quality score per mistake type
    avg_score_by_mistake = df_mistakes.groupby('mistake')['quality_score'].mean().sort_values()
    avg_score_by_mistake.plot(kind='barh', ax=axes[1], color=sns.color_palette('RdYlGn', len(avg_score_by_mistake)))
    axes[1].set_title('Average Quality Score by Mistake Type')
    axes[1].set_xlabel('Average Score')
    axes[1].set_xlim(0, 5)
else:
    axes[0].text(0.5, 0.5, 'No mistakes found', ha='center', va='center')
    axes[1].text(0.5, 0.5, 'No mistakes found', ha='center', va='center')

plt.tight_layout()
plt.show()

## 7. Intent Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Intent distribution
intent_counts = df['intent'].value_counts()
intent_counts.plot(kind='barh', ax=axes[0], color=sns.color_palette('Set2', len(intent_counts)))
axes[0].set_title('Intent Distribution (from analysis)')
axes[0].set_xlabel('Count')

# Average quality score by intent
avg_score_by_intent = df.groupby('intent')['quality_score'].mean().sort_values()
avg_score_by_intent.plot(kind='barh', ax=axes[1], color=sns.color_palette('RdYlGn', len(avg_score_by_intent)))
axes[1].set_title('Average Quality Score by Intent')
axes[1].set_xlabel('Average Score')
axes[1].set_xlim(0, 5)

plt.tight_layout()
plt.show()

## 8. Confidence Score Analysis

Analyze model confidence in satisfaction predictions and its calibration.

In [ ]:
if 'confidence' in df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Confidence distribution
    axes[0].hist(df['confidence'], bins=20, edgecolor='black', alpha=0.7, color='#3498db')
    axes[0].set_title('Confidence Score Distribution')
    axes[0].set_xlabel('Confidence')
    axes[0].set_ylabel('Count')
    axes[0].axvline(df['confidence'].mean(), color='red', linestyle='--', label=f"Mean: {df['confidence'].mean():.2f}")
    axes[0].legend()

    # Confidence by satisfaction level
    sat_order = ['satisfied', 'neutral', 'unsatisfied']
    sat_colors = ['#2ecc71', '#f1c40f', '#e74c3c']
    for level, color in zip(sat_order, sat_colors):
        subset = df[df['satisfaction'] == level]['confidence']
        if len(subset) > 0:
            axes[1].hist(subset, bins=15, alpha=0.5, label=f"{level} (avg={subset.mean():.2f})",
                        color=color, edgecolor='black')
    axes[1].set_title('Confidence by Satisfaction Level')
    axes[1].set_xlabel('Confidence')
    axes[1].set_ylabel('Count')
    axes[1].legend()

    # Confidence vs quality score (box plot)
    df.boxplot(column='confidence', by='quality_score', ax=axes[2])
    axes[2].set_title('Confidence by Quality Score')
    axes[2].set_xlabel('Quality Score')
    axes[2].set_ylabel('Confidence')
    plt.suptitle('')

    plt.tight_layout()
    plt.show()

    print(f"Average confidence: {df['confidence'].mean():.3f}")
    print(f"Std dev: {df['confidence'].std():.3f}")
    print(f"Min: {df['confidence'].min():.3f}, Max: {df['confidence'].max():.3f}")

    # Confidence calibration: hidden dissatisfaction cases
    if len(hidden) > 0 and 'confidence' in hidden.columns:
        print(f"\nHidden dissatisfaction — avg confidence: {hidden['confidence'].mean():.3f}")
        print(f"Non-hidden — avg confidence: {non_hidden['confidence'].mean():.3f}")
else:
    print("'confidence' field not found in analysis results. Run analyze.py with updated version.")

## 9. Validation Warnings Analysis

Analyze post-validation corrections applied to LLM analysis results (7 consistency rules).

In [ ]:
if 'validation_warnings' in df.columns:
    df['num_warnings'] = df['validation_warnings'].apply(len)
    corrected = df[df['num_warnings'] > 0]

    print(f"Dialogs with validation corrections: {len(corrected)}/{len(df)} ({len(corrected)/len(df)*100:.1f}%)")
    print(f"Total corrections applied: {df['num_warnings'].sum()}")

    if len(corrected) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Corrections by case type
        corrections_by_case = corrected.groupby('case_type').size()
        corrections_by_case.plot(kind='bar', ax=axes[0], color=sns.color_palette('Set2'), edgecolor='black')
        axes[0].set_title('Validation Corrections by Case Type')
        axes[0].set_ylabel('Count')
        axes[0].set_xticklabels(corrections_by_case.index, rotation=45)

        # Extract rule names from warnings
        all_warnings = []
        for warnings_list in corrected['validation_warnings']:
            for w in warnings_list:
                if 'corrected:' in w:
                    rule_part = w.split('(')[1].rstrip(')') if '(' in w else w
                    all_warnings.append(rule_part)
                else:
                    all_warnings.append(w[:40])

        if all_warnings:
            warning_counts = pd.Series(all_warnings).value_counts()
            warning_counts.plot(kind='barh', ax=axes[1], color='#e74c3c', alpha=0.8)
            axes[1].set_title('Most Common Validation Corrections')
            axes[1].set_xlabel('Count')
        else:
            axes[1].text(0.5, 0.5, 'No parseable warnings', ha='center', va='center')

        plt.tight_layout()
        plt.show()

        # Show corrected cases details
        print("\nCorrected cases:")
        for _, row in corrected.iterrows():
            print(f"  {row['chat_id']} ({row['case_type']}): {row['validation_warnings']}")
else:
    print("'validation_warnings' field not found in analysis results.")

## 10. Summary & Conclusions

In [ ]:
print("=" * 60)
print("DATASET ANALYSIS SUMMARY")
print("=" * 60)

print(f"\n1. DATASET SIZE: {len(df)} dialogs")
print(f"   Average dialog length: {df['num_messages'].mean():.1f} messages")

print("\n2. QUALITY SCORES:")
print(f"   Average: {df['quality_score'].mean():.2f} / 5.00")
print(f"   Median: {df['quality_score'].median():.0f}")
print(f"   Score 4-5 (good): {(df['quality_score'] >= 4).sum()} ({(df['quality_score'] >= 4).sum()/len(df)*100:.1f}%)")
print(f"   Score 1-2 (poor): {(df['quality_score'] <= 2).sum()} ({(df['quality_score'] <= 2).sum()/len(df)*100:.1f}%)")

print("\n3. SATISFACTION:")
for level in ['satisfied', 'neutral', 'unsatisfied']:
    count = (df['satisfaction'] == level).sum()
    print(f"   {level}: {count} ({count/len(df)*100:.1f}%)")

print("\n4. HIDDEN DISSATISFACTION:")
hidden_total = df['has_hidden_dissatisfaction'].sum()
if hidden_total > 0:
    hidden_detected = (df['has_hidden_dissatisfaction'] & (df['satisfaction'] != 'satisfied')).sum()
    print(f"   Intended: {hidden_total} dialogs")
    print(f"   Detected (unsatisfied/neutral): {hidden_detected} ({hidden_detected/hidden_total*100:.1f}%)")
    print(f"   Missed (marked satisfied): {hidden_total - hidden_detected}")
else:
    print("   No hidden dissatisfaction cases in dataset")

print("\n5. AGENT MISTAKES:")
print(f"   Dialogs with mistakes: {chats_with_mistakes} ({chats_with_mistakes/len(df)*100:.1f}%)")
if len(df_mistakes) > 0:
    top_mistake = df_mistakes['mistake'].value_counts().index[0]
    print(f"   Most common mistake: {top_mistake}")
    print(f"   Total mistake types found: {df_mistakes['mistake'].nunique()}")

print("\n6. ANOMALIES (post-validation):")
print(f"   High score + unsatisfied: {len(high_score_unsatisfied)}")
print(f"   Low score + satisfied: {len(low_score_satisfied)}")
print(f"   No mistakes + low score: {len(no_mistakes_low)}")

if 'confidence' in df.columns:
    print("\n7. CONFIDENCE:")
    print(f"   Average: {df['confidence'].mean():.3f}")
    print(f"   Range: [{df['confidence'].min():.3f}, {df['confidence'].max():.3f}]")

if 'validation_warnings' in df.columns:
    num_corrected = (df['validation_warnings'].apply(len) > 0).sum()
    print(f"\n8. VALIDATION CORRECTIONS: {num_corrected}/{len(df)} dialogs corrected")

print("\n" + "=" * 60)